# Can an auxiliary encoder correct a strong predictor's errors?

Encoder swaps compare complete single-input and fusion predictors. This experiment asks a more focused question: once a strong encoder and its fitted prediction head provide an **anchor**, can the other representations predict useful corrections? The anchor remains fixed while each correction model learns. This implements the regression branch of Mina's auxiliary error-correction proposal, with explicit project choices for folds, models and controls. [Mina's plan, section A.2](../../docs/Biological_Fusion_Plan.docx); [execution plan, B6–B8](../../docs/research-execution-plan.md#b7--implement-anchor-correction-and-verify-its-mechanics).

The three targets remain distinct: CoV-Vac measures chemical degradation of designed RNA, fungal expression uses its supplied expression target, and mRNA stability uses its supplied stability score. A numerical gain means better prediction of that particular target; it does not identify a causal biological mechanism or turn a computationally translated RNA fragment into an established protein product. [Dataset targets and interpretation](../../docs/methods/data-and-targets.md).

Both the original NT-v2 / RNA-FM / ESM-2 combination and the mRNA-FM replacement are retained. Stability uses the separately saved exact-sequence grouped split. Prior benchmark results informed this panel, so this is an exploratory investigation, not a newly untouched confirmation dataset.

## Setup

The default mode reads local evidence and coverage without opening benchmark CSVs, token caches or models. Training requires an explicit Newton execution configuration; statistics have their own switch. [Environment setup](../../README.md#setup) and [Stage 4 runbook](RUNBOOK.md) describe the shared preparation route.

In [1]:
from contextlib import ExitStack
from dataclasses import asdict, replace
import hashlib
import json
from pathlib import Path
import sys

import numpy as np  # Import before PyTorch in the Windows environment.
import pandas as pd
import torch
from IPython.display import display

In [2]:
# Locate the Code directory from either a repository-root or notebook working directory.
CODE_DIR = next(
    candidate / "Code" if (candidate / "Code" / "shared_code").is_dir() else candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "Code" / "shared_code").is_dir() or (candidate / "shared_code").is_dir()
)
PROJECT_DIR = CODE_DIR.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from shared_code.fusion import correction_experiment as correction
from shared_code.fusion import run_identity, study_inputs, study_selection, swap_reporting, token_embeddings, train

In [3]:
READ_SAVED_RESULTS = True
RUN_TRAINING = False
RUN_ANALYSIS = False
RUN_DIAGNOSTICS = False  # Saved validation/OOF predictions and fit histories only.
RUN_INPUT_AUDIT = False  # Local CSV + existing split; no token arrays or models.
RUN_CACHE_AUDIT = False  # Bounded token-array inspection; separate from input audit.
AUDIT_ROWS_PER_PARTITION = 32  # Inspection budget, not a training/sample-size decision.
AUDIT_SAMPLE_SEED = 0
DEVICE = "cpu"
PRODUCER = None  # The Newton launcher supplies observed environment/source provenance.

BENCHMARK = "cov_vac_degradation"
BENCHMARKS = ("cov_vac_degradation", "fungal_expression", "mrna_stability")
SEED = 0
TRAINING_SEEDS = (0, 1, 2)
SELECTION_SEED = 0
N_FOLDS = 5
SPLIT_SEED = 42
INNER_SEED = 42
INNER_STOP_FRACTION = 0.20
CONTROL_SEED = 0
BOOTSTRAP_DRAWS = 2000
BOOTSTRAP_SEED = 0
COMPARE_CORRECTION_FAMILIES = False  # Optional prospective contrast; default panel omits it.
USE_NEWTON_ANCHORS = True  # Compatible singles use existing best weights; never auto-refit.
RIDGE_ALPHAS = (1e-3, 0.01, 0.1, 1.0, 10.0)

DATA_ROOT = PROJECT_DIR / "datasets"
CACHE_ROOT = CODE_DIR / "stage3" / "embeddings"
GROUPED_SPLIT_PATH = CODE_DIR / "stage4" / "results" / "grouped_stability_v1" / "split.json"
GROUPED_REFERENCE_ROOT = GROUPED_SPLIT_PATH.parent
REFERENCE_ROOT = CODE_DIR / "stage4" / "results" / "newton_h100_seeds012"
RESULTS_ROOT = CODE_DIR / "stage4" / "results" / "anchor_correction"

# Keep the established BLF training recipe; the job changes only its training seed.
BASE_CONFIG = train.TrainConfig(tconv="equation", seed=SELECTION_SEED)

In [4]:
# This executed copy enables only the user-authorized inspection.
BENCHMARK = 'mrna_stability'
READ_SAVED_RESULTS = True
RUN_INPUT_AUDIT = True
RUN_CACHE_AUDIT = False
RUN_TRAINING = RUN_ANALYSIS = RUN_DIAGNOSTICS = False
DEVICE = "cpu"
def _outside_inspection(*args, **kwargs):
    raise RuntimeError("Training, extraction, model inference and bootstrap are outside this inspection.")
train.run = _outside_inspection
train.seed_averaged_bootstrap = _outside_inspection
token_embeddings.extract = _outside_inspection
swap_reporting.calculate = _outside_inspection
# Inspect the agreed candidate; never publish the canonical training split.
from shared_code import datasets
from shared_code.fusion import grouped_stability
import hashlib
import json
import tempfile
_candidate_dir = tempfile.TemporaryDirectory(prefix="biofusion-split-inspection-")
GROUPED_SPLIT_PATH = Path(_candidate_dir.name) / "split.json"
_rows = datasets.blf_benchmark_rows(BENCHMARK, data_dir=DATA_ROOT)
_source = DATA_ROOT / datasets.DATASETS[BENCHMARK].path
_, _candidate = grouped_stability.build_split(
    _rows, source_sha256=hashlib.sha256(_source.read_bytes()).hexdigest(), split_seed=SPLIT_SEED,
)
GROUPED_SPLIT_PATH.write_text(json.dumps(_candidate), encoding="utf-8")
_evidence_candidate = PROJECT_DIR / "Code/stage4/results/decision_inspections/20261117_pre_newton/correction/stability_candidate/candidate-split.json"
if _evidence_candidate.exists():
    assert json.loads(_evidence_candidate.read_text()) == _candidate
else:
    with _evidence_candidate.open("x", encoding="utf-8") as _handle:
        json.dump(_candidate, _handle, indent=2)
print("Candidate grouped split:", _candidate["fingerprint"], "; canonical split not published")
print("Authorized inspection:", BENCHMARK, 'folds', "on CPU")


Candidate grouped split: 793991aa9bec95e23d6a2c055c61bb5e09f718bf57418f628d6c48a7edce5d40 ; canonical split not published
Authorized inspection: mrna_stability folds on CPU


Five OOF folds and a fixed seed-0 validation-selected anchor are agreed project choices. The inner stopping fraction of 20%, its hash assignment and seed 42 are explicit project settings, not values prescribed by Mina. Each fold restores the best model fitted on its inner-fit rows; it does not subsequently refit on its stopping rows. The final anchor uses all outer-training rows and stops on outer validation. Neither predictor is refitted on train plus outer validation.

The ridge grid is explicit and preserves the declared order for ties. Ridge chooses regularization using validation mean squared error of the corrected prediction. Neural corrections retain the inherited training budget and choose checkpoints by corrected validation Spearman. These selection criteria differ deliberately and are recorded with their results.

The inherited BLF budget and ridge grid provide a familiar starting comparison, not demonstrated optimal settings for correction. Mina specifies neither those numerical values nor these two correction families. Keeping them explicit lets the diagnostics support retaining them or identify a concrete reason for a separately reviewed change; reading a weak result does not authorize tuning against the test set.

In [5]:
STUDY = "fixed-anchor-regression-correction-v1"
BUNDLES = {
    "original": {"dna": "ntv2_100m", "rna": "rnafm", "prot": "esm2_8m"},
    "mrnafm": {"dna": "ntv2_100m", "rna": "mrnafm", "prot": "esm2_8m"},
}
CANDIDATES = {
    "ntv2_native": {"modality": "dna", "layout": "native", "bundle": "original"},
    "ntv2_grid": {"modality": "dna", "layout": "codon", "bundle": "original"},
    "rnafm_native": {"modality": "rna", "layout": "native", "bundle": "original"},
    "rnafm_grid": {"modality": "rna", "layout": "codon", "bundle": "original"},
    "mrnafm_native": {"modality": "rna", "layout": "native", "bundle": "mrnafm"},
    "esm2_native": {"modality": "prot", "layout": "native", "bundle": "original"},
}
ELIGIBLE = {
    "original": ("ntv2_native", "ntv2_grid", "rnafm_native", "rnafm_grid", "esm2_native"),
    "mrnafm": ("ntv2_native", "ntv2_grid", "mrnafm_native", "esm2_native"),
}
CONTROL_KINDS = ("real", "shuffle", "random")
CORRECTION_FAMILIES = ("ridge", "textcnn")

In [6]:
def require_training_permission():
    if READ_SAVED_RESULTS or not RUN_TRAINING:
        raise RuntimeError("Enable the explicit training mode before requesting fits.")
    if BENCHMARK not in BENCHMARKS or SEED not in TRAINING_SEEDS:
        raise ValueError("Choose one declared regression benchmark and training seed.")
    if str(DEVICE) != "cuda" or not torch.cuda.is_available():
        raise RuntimeError("This production batch requires the allocated CUDA GPU; no CPU fallback.")
    if not isinstance(PRODUCER, dict) or not PRODUCER:
        raise RuntimeError("Provide the launcher's observed producer/source record.")
    if RUN_ANALYSIS or RUN_DIAGNOSTICS or RUN_INPUT_AUDIT or RUN_CACHE_AUDIT:
        raise ValueError("Training and downstream analysis are separate execution batches.")
    protected = (CACHE_ROOT, CODE_DIR / "stage3" / "results",
                 CODE_DIR / "stage4" / "results" / "phaseA",
                 CODE_DIR / "stage4" / "results" / "phaseB",
                 CODE_DIR / "stage4" / "results" / "newton_h100_seeds012")
    root = Path(RESULTS_ROOT).resolve()
    for folder in protected:
        folder = Path(folder).resolve()
        if root == folder or root in folder.parents or folder in root.parents:
            raise ValueError(f"New correction results overlap preserved evidence: {folder}")


if not READ_SAVED_RESULTS and not RUN_TRAINING:
    raise ValueError("Choose saved-result reading or explicitly authorized training.")
print("Reading saved correction evidence." if READ_SAVED_RESULTS else
      "Training requested; GPU, provenance and destination checks precede data loading.")

if any((RUN_INPUT_AUDIT, RUN_CACHE_AUDIT, RUN_DIAGNOSTICS)):
    if not READ_SAVED_RESULTS or RUN_TRAINING or RUN_ANALYSIS:
        raise ValueError("Decision-support inspections require a separate read-results session.")
    if sum(map(bool, (RUN_INPUT_AUDIT, RUN_CACHE_AUDIT, RUN_DIAGNOSTICS))) != 1:
        raise ValueError("Choose one diagnostic workload at a time.")

Reading saved correction evidence.


---

## What evidence is already available?

A completed fit and its prediction summary are different artifacts. Coverage below counts readable saved summaries and reports malformed entries separately. It does not infer completion from elapsed epochs, recompute scores, fill missing runs or train missing runs. Compatible Newton single-encoder fits can supply unchanged validation scores and test predictions; their best weights provide new validation predictions without retraining. Missing best weights hold that route. Grouped stability reuses the separately completed grouped-panel singles, including NT-grid. Supplied-split NT-grid and every OOF anchor require new fits.

In [7]:
def saved_coverage(root):
    rows = []
    for path in sorted(Path(root).glob("**/predictions.json")):
        try:
            record = json.loads(path.read_text(encoding="utf-8"))
            if record.get("contract") != STUDY:
                raise ValueError("Different result contract")
            family = record["kind"]
            identity = record.get("inference_identity", record["identity"])
            for split in ("val", "test"):
                entry = record["predictions"][split]
                values = np.asarray(entry["values"])
                if values.shape != (len(entry["row_ids"]),) or not np.isfinite(values).all():
                    raise ValueError(f"Invalid {split} prediction array")
            rows.append({"path": str(path.relative_to(root)), "family": family,
                         "role": identity.get("inputs", {}).get("role", "correction"),
                         "status": "saved", "issue": ""})
        except (KeyError, TypeError, ValueError, OSError) as error:
            rows.append({"path": str(path.relative_to(root)), "family": "unknown",
                         "role": "unknown", "status": "review", "issue": str(error)})
    return pd.DataFrame(rows, columns=("path", "family", "role", "status", "issue"))

In [8]:
coverage = saved_coverage(RESULTS_ROOT)
if coverage.empty:
    print("No saved correction summaries found at the selected results root.")
else:
    display(coverage)

No saved correction summaries found at the selected results root.


## Learn corrections from predictions made without the training example

An in-sample anchor may already fit a training row unusually well. Its apparent error is therefore a poor target for measuring what an auxiliary can contribute. Instead, every outer-training sequence group is held out from one anchor fit and its inner stopping set. That fit supplies the group's out-of-fold (OOF) prediction. [Mina's plan, A.2](../../docs/Biological_Fusion_Plan.docx).

$$
e_i^{\mathrm{OOF}} = y_i - \hat y_{a,i}^{\mathrm{OOF}},
\qquad
\hat y_i = \hat y_{a,i}^{\mathrm{final}} + q(h_i^{\mathrm{aux}}).
$$

Here $y_i$ is the benchmark target; $\hat y_{a,i}^{\mathrm{OOF}}$ is the prediction from the fold anchor; $q$ is the learned auxiliary correction; and $\hat y_{a,i}^{\mathrm{final}}$ is the final anchor prediction on validation or test. The corrected predictor and anchor-only comparator use exactly the same final anchor artifact. For example, an anchor prediction of 0.4 plus a learned correction of −0.1 gives 0.3; these values illustrate arithmetic, not results.

Anchor architecture selection uses seed-0 outer validation before OOF construction. The OOF exclusion claim is conditional on that selected configuration; it does not claim that architecture choice was nested inside every fold. Training targets cannot choose their own fold's preprocessing, epoch or stopping criterion. The implementation tests row and group exclusion, while provenance records identify the actual fit and stopping rows.

[Stacked generalization](../../docs/sources.md#wolpert-stacked-generalization) supplies background for using withheld first-stage predictions. Our grouping, additive correction and control protocol are project choices; this source does not establish Mina's historical motivation.


In [9]:
def benchmark_root(benchmark):
    return Path(RESULTS_ROOT) / benchmark


def anchor_folder(benchmark, candidate, seed):
    return benchmark_root(benchmark) / "anchors" / candidate / f"seed{seed}"


def selection_evidence_roots():
    # Named roots survive a Newton-to-local return without copying or relabeling fits.
    return {"study": Path(RESULTS_ROOT), "newton": Path(REFERENCE_ROOT),
            "grouped": Path(GROUPED_REFERENCE_ROOT)}


def selection_settings(bundle):
    return {"study": STUDY, "bundle": bundle, "eligible": list(ELIGIBLE[bundle]),
            "selection_seed": SELECTION_SEED, "base_config": asdict(BASE_CONFIG),
            "reuse_newton_anchors": USE_NEWTON_ANCHORS}


def selection_context(bundle, split_identity):
    return {"settings": selection_settings(bundle), "outer_split": split_identity}


def read_selection(benchmark, bundle, split_identity=None):
    path = benchmark_root(benchmark) / f"selection_{bundle}.json"
    recorded = json.loads(path.read_text(encoding="utf-8"))
    context = recorded["context"]
    if context["settings"] != selection_settings(bundle):
        raise ValueError("Saved anchor panel/recipe differs from these settings.")
    if split_identity is not None and context != selection_context(bundle, split_identity):
        raise ValueError("Saved anchor selection belongs to another outer split.")
    return study_selection.read_anchor(path, context=context, evidence_roots=selection_evidence_roots())

In [10]:
def save_decision_report(kind, analysis, inputs):
    # A new rule/input identity gets a new file; an existing report is never replaced.
    notebook = json.loads((CODE_DIR / "stage4" / "anchor_correction.ipynb").read_text(encoding="utf-8"))
    identity = {
        "schema": "fusion.anchor-decision-support.v1", "kind": kind, "benchmark": BENCHMARK,
        "inputs": inputs, "source_sha256": correction._source_hashes(),
        "notebook_code_sha256": correction._digest([
            "".join(cell["source"]) for cell in notebook["cells"] if cell["cell_type"] == "code"]),
    }
    record = {"identity": identity, "analysis": analysis,
              "analysis_sha256": correction._digest(analysis)}
    path = benchmark_root(BENCHMARK) / "analyses" / f"decision-support-{kind}-{correction._digest(identity)[:16]}.json"
    if path.exists():
        existing = json.loads(path.read_text(encoding="utf-8"))
        if any(existing.get(key) != value for key, value in record.items()):
            raise ValueError("A decision-support report differs under the same identity; preserve it.")
    else:
        correction._publish(path, {**record, "producer": run_identity.producer_record(
            device="cpu", repository=PROJECT_DIR, packages=("numpy", "pandas"))})
    return path


def read_decision_reports(kind):
    reports = []
    for path in sorted((benchmark_root(BENCHMARK) / "analyses").glob(f"decision-support-{kind}-*.json")):
        record = json.loads(path.read_text(encoding="utf-8"))
        identity = record.get("identity", {})
        expected = f"decision-support-{kind}-{correction._digest(identity)[:16]}.json"
        if (identity.get("schema") != "fusion.anchor-decision-support.v1"
                or identity.get("benchmark") != BENCHMARK or identity.get("kind") != kind
                or path.name != expected or not identity.get("inputs")
                or record.get("analysis_sha256") != correction._digest(record.get("analysis"))):
            raise ValueError(f"Incomplete or changed decision-support report: {path}")
        reports.append((path, record))
    return reports

### Check whether the agreed folds have usable support

Mina requires group-aware OOF predictions and independent model selection. Five folds, a fixed seed-0 anchor and 20% fold-local stopping are our agreed implementation choices. Holding them fixed makes the correction question manageable; it does not establish that every fold has enough independent examples or target variation. The following input audit inspects the existing assignment without fitting an anchor, searching seeds or changing a split.

Read rows and groups together: many observations from one sequence still supply one independent sequence group. A constant stopping target makes validation Spearman undefined; strong group concentration or uneven target/length support warrants review. Otherwise these summaries can support retaining the agreed design. There is no automatic balance threshold or new exclusion rule. The audit reads only the local benchmark CSV and, for stability, the saved grouped split. Saved reports describe their recorded input snapshot; reading them does not re-audit today's data.

In [11]:
def inspect_fold_inputs():
    rows, _, groups, positions, split_identity = study_inputs.benchmark_rows(
        BENCHMARK, DATA_ROOT, stability_split=GROUPED_SPLIT_PATH,
    )
    plans = correction.fold_plan(positions["train"], groups, n_splits=N_FOLDS,
                                 seed=SPLIT_SEED, inner_seed=INNER_SEED,
                                 stop_fraction=INNER_STOP_FRACTION)
    context = {"row_ids": tuple(rows.index), "groups": groups,
               "train_positions": positions["train"], "validation_positions": positions["val"],
               "test_positions": positions["test"]}
    table = correction.evidence.fold_diagnostics(
        plans, targets=rows["Value"].to_numpy(), lengths=rows["length"].to_numpy(), **context,
    )
    inputs = {"outer_split": split_identity, "folds": [asdict(plan) for plan in plans],
              "settings": {"n_folds": N_FOLDS, "seed": SPLIT_SEED, "inner_seed": INNER_SEED,
                           "stop_fraction": INNER_STOP_FRACTION},
              "targets": run_identity.array_record(rows["Value"].to_numpy()),
              "lengths": run_identity.array_record(rows["length"].to_numpy())}
    return save_decision_report("folds", {"folds": table}, inputs)

In [12]:
if RUN_INPUT_AUDIT:
    print(f"Saved fold inspection: {inspect_fold_inputs()}")
for path, report in read_decision_reports("folds"):
    print(f"Recorded input snapshot: {path.name}")
    display(pd.DataFrame(report["analysis"]["folds"]))

Saved fold inspection: D:\academic\UCF\2026_fall\Capstone Project CAP6942\Research\genomics-multimodal-fusion\Code\stage4\results\anchor_correction\mrna_stability\analyses\decision-support-folds-a98018b61aaa682c.json
Recorded input snapshot: decision-support-folds-a98018b61aaa682c.json


,constant_target,fold,groups,largest_group_row_fraction,median_length,role,rows,target_max,target_min,target_sd,target_unique
0,False,0,5010,0.000940,663.0,fit,10635,2.889911,-3.109433,0.977191,10615
1,False,0,1265,0.001782,648.0,stop,2806,2.698830,-2.930751,0.981905,2803
2,False,0,1679,0.001429,669.0,predict,3499,2.800427,-2.945895,0.987022,3497
3,False,1,5129,0.000555,663.0,fit,10815,2.889911,-2.962063,0.985634,10805
4,False,1,1271,0.002901,666.0,stop,2758,2.800427,-2.930751,0.974142,2755
5,False,1,1554,0.002970,663.0,predict,3367,2.832854,-3.109433,0.966623,3357
6,False,2,5102,0.000918,666.0,fit,10890,2.832854,-2.960427,0.981625,10873
7,False,2,1271,0.002169,649.5,stop,2766,2.664315,-3.109433,0.966156,2761
8,False,2,1581,0.002436,663.0,predict,3284,2.889911,-2.962063,0.986090,3281
9,False,3,5106,0.000931,666.0,fit,10743,2.889911,-3.109433,0.985478,10725


In [13]:
def fit_anchor(opened, benchmark, candidate, seed):
    spec = CANDIDATES[candidate]
    streams, records, groups, positions, split_identity = opened[spec["bundle"]]
    if USE_NEWTON_ANCHORS and benchmark == "mrna_stability":
        return correction.reuse_grouped_anchor(
            streams, reference_root=GROUPED_REFERENCE_ROOT, candidate=candidate,
            config=replace(BASE_CONFIG, seed=seed), row_ids=tuple(streams.rows.index), groups=groups,
            split_positions=positions, cache_records=records,
            output_dir=anchor_folder(benchmark, candidate, seed),
            input_contract={"study": STUDY, "candidate": candidate, "outer_split": split_identity},
            device=DEVICE, producer=PRODUCER,
        )
    if USE_NEWTON_ANCHORS and benchmark != "mrna_stability" and candidate != "ntv2_grid":
        return correction.reuse_newton_anchor(
            streams, reference_root=REFERENCE_ROOT, bundle=spec["bundle"], modality=spec["modality"],
            layout=spec["layout"], config=replace(BASE_CONFIG, seed=seed),
            row_ids=tuple(streams.rows.index), groups=groups, split_positions=positions,
            cache_records=records, output_dir=anchor_folder(benchmark, candidate, seed),
            input_contract={"study": STUDY, "candidate": candidate, "outer_split": split_identity},
            device=DEVICE, producer=PRODUCER,
        )
    return correction.run_neural(
        streams, split_positions=positions, row_ids=tuple(streams.rows.index), groups=groups,
        modalities=(spec["modality"],), layout=spec["layout"], offsets=np.zeros(len(streams.rows)),
        config=replace(BASE_CONFIG, seed=seed), output_dir=anchor_folder(benchmark, candidate, seed),
        cache_records=records, input_contract={"study": STUDY, "candidate": candidate,
                                               "outer_split": split_identity},
        device=DEVICE, producer=PRODUCER, role="anchor", control={"kind": "real"},
    )


def prepare_selections(opened, benchmark):
    # Candidate fits shared by the two bundles have one canonical encoder context.
    if SEED == SELECTION_SEED:
        fitted = {candidate: fit_anchor(opened, benchmark, candidate, SELECTION_SEED)
                  for candidate in CANDIDATES}
        fit_paths = {
            key: (correction.reference_artifacts(artifact.record,
                GROUPED_REFERENCE_ROOT if benchmark == "mrna_stability" else REFERENCE_ROOT)["fit"]
                  if artifact.record["kind"] == "reference_anchor" else artifact.path.parent / "fit.json")
            for key, artifact in fitted.items()
        }
        for bundle in BUNDLES:
            candidates = [{"key": key, **CANDIDATES[key],
                           "path": fit_paths[key]}
                          for key in ELIGIBLE[bundle]]
            study_selection.select_anchor(
                candidates, path=benchmark_root(benchmark) / f"selection_{bundle}.json",
                context=selection_context(bundle, opened[bundle][4]), evidence_roots=selection_evidence_roots(),
            )
    return {bundle: read_selection(benchmark, bundle, opened[bundle][4]) for bundle in BUNDLES}

### How decisive is the fixed anchor choice?

The selection records retain every candidate's validation score. This readout reports the existing winner and its margin above the runner-up; it does not choose another winner. A small margin makes the conclusions more dependent on this particular seed-0 choice. A clear margin supports the fixed-anchor comparison, but does not establish that the encoder is universally strongest. Ties retain declared candidate order. Comparable validation ranks from later training seeds could be examined separately once that full candidate panel exists; a partial panel must not stand in for it.

`RUN_DIAGNOSTICS` calculates the margin from saved validation scores and saves its own report; default reading displays that report without rescoring. The seed-0 choice remains fixed across the three training seeds, and final-test observations cannot revise it.

In [14]:
if RUN_DIAGNOSTICS:
    selection_rows, selection_sources = [], {}
    for bundle in BUNDLES:
        selection_path = benchmark_root(BENCHMARK) / f"selection_{bundle}.json"
        if not selection_path.exists():
            print(f"{bundle}: anchor selection not yet saved.")
            continue
        selected = read_selection(BENCHMARK, bundle)
        record = json.loads(selection_path.read_text(encoding="utf-8"))
        ranked = sorted(record["candidates"], key=lambda item: item["validation_score"], reverse=True)
        margin = ranked[0]["validation_score"] - ranked[1]["validation_score"]
        selection_rows.extend({"bundle": bundle, "key": item["key"], "modality": item["modality"],
                               "layout": item["layout"], "validation_score": item["validation_score"],
                               "selected": item["key"] == selected["key"], "winner_margin": margin}
                              for item in record["candidates"])
        selection_sources[bundle] = {"selection_sha256": selected["selection_sha256"],
                                      "candidates": record["candidates"]}
    if selection_sources:
        save_decision_report("selection", {"candidates": selection_rows}, {"sources": selection_sources})
for path, report in read_decision_reports("selection"):
    print(f"Recorded selection snapshot: {path.name}")
    display(pd.DataFrame(report["analysis"]["candidates"]))

## Ask the auxiliary question with two correction families

Each bundle contributes the two non-anchor streams separately and together. Ridge receives mean-pooled native representations, with joint features concatenated; its scaler is fitted only on correction-training rows. TextCNN receives each single auxiliary at native token resolution, while joint auxiliaries are placed on the existing codon grid and concatenated. Its output is added to the fixed anchor prediction before regression loss and validation scoring.

Consequently, ridge versus TextCNN also changes pooling, positional access and capacity; it is not a pure test of nonlinearity. Joint TextCNN includes the existing trainable DNA grid projection where needed. The neural loss on `anchor + q` against the original target is algebraically the residual MSE, while corrected validation Spearman determines its best epoch.

In [15]:
def auxiliary_sets(anchor_modality):
    others = tuple(modality for modality in ("dna", "rna", "prot") if modality != anchor_modality)
    return {others[0]: (others[0],), others[1]: (others[1],), "+".join(others): others}


def control_specs(streams, positions, groups, modalities, seed):
    # Constant strata mean exchangeable sequence groups within each outer partition.
    # The group derangement moves all joint streams and their lengths together.
    mapping = np.arange(len(streams.rows))
    strata = {}
    for split, indices in positions.items():
        strata[split] = {str(groups[index]): "within-partition" for index in indices}
        shuffled = correction.group_permutation(
            groups, indices, strata_by_group=strata[split], seed=seed,
        )
        mapping[indices] = shuffled[indices]
    statistics = correction.training_random_statistics(streams, positions["train"], modalities)
    return {
        "real": {"kind": "real"},
        "shuffle": {"kind": "shuffle", "row_map": mapping.tolist(), "seed": seed, "strata": strata},
        "random": {"kind": "random", "seed": seed, "statistics": statistics},
    }

### Does the shuffled control actually break the useful correspondence?

Mina's shuffled-input control motivates reassignment, while the within-partition group cycle and constant stratum are our implementation choices. Groups move together and joint auxiliaries share a donor. Different nucleotide groups can still encode identical proteins, so a derangement of group IDs alone does not establish a changed protein representation. Unequal repeated-group sizes can also change the row-weighted frequency of donor features.

The optional cache audit reports all-row length changes and donor-exposure total variation (zero means unchanged group frequencies), then exact feature equality and pooled norms for a bounded, hash-selected sample. It uses no labels to select the sample. A high unchanged-feature fraction or substantial distribution shift would challenge the control's interpretation; it does not trigger another shuffle. This is a sample inspection, not a population estimate. Cache verification may read shard bytes before the bounded feature sample. Random-control training moments are inspected later from saved fit metadata rather than recalculating them during this audit.

In [16]:
def inspect_cached_controls():
    if type(AUDIT_ROWS_PER_PARTITION) is not int or AUDIT_ROWS_PER_PARTITION <= 0:
        raise ValueError("Choose a positive explicit feature-inspection budget.")
    audits, identities = [], {}
    with ExitStack() as resources:
        verification = token_embeddings.CacheVerification()
        for bundle, encoders in BUNDLES.items():
            streams, records, groups, positions, split_identity = resources.enter_context(
                study_inputs.open_streams(BENCHMARK, encoders, DATA_ROOT, CACHE_ROOT,
                                         stability_split=GROUPED_SPLIT_PATH, verification=verification))
            mapping = np.arange(len(streams.rows))
            strata, sample = {}, []
            for split, indices in positions.items():
                strata[split] = {str(groups[index]): "within-partition" for index in indices}
                shuffled = correction.group_permutation(
                    groups, indices, strata_by_group=strata[split], seed=CONTROL_SEED + SEED)
                mapping[indices] = shuffled[indices]
                ordered = sorted(indices, key=lambda index: correction._digest(
                    [AUDIT_SAMPLE_SEED, str(groups[index]), int(index)]))
                sample.extend(int(index) for index in ordered[:AUDIT_ROWS_PER_PARTITION])
            control = {"kind": "shuffle", "row_map": mapping.tolist(),
                       "seed": CONTROL_SEED + SEED, "strata": strata}
            audit = correction.inspect_shuffle(streams, groups=groups, split_positions=positions,
                                               control=control, sample_positions=sample)
            audits.append({"bundle": bundle, **audit})
            identities[bundle] = {"outer_split": split_identity, "caches": records,
                                  "mapping_sha256": correction._digest(mapping.tolist()),
                                  "sample_positions": sample}
    return save_decision_report("controls", {"bundles": audits}, {
        "bundles": identities, "training_seed": SEED, "control_seed": CONTROL_SEED + SEED,
        "sample_seed": AUDIT_SAMPLE_SEED, "rows_per_partition": AUDIT_ROWS_PER_PARTITION})

In [17]:
if RUN_CACHE_AUDIT:
    print(f"Saved bounded control inspection: {inspect_cached_controls()}")
for path, report in read_decision_reports("controls"):
    print(f"Recorded cache snapshot: {path.name}")
    for audit in report["analysis"]["bundles"]:
        print(audit["bundle"])
        display(pd.DataFrame(audit["metadata"]))
        sample = pd.DataFrame(audit["sample"])
        display(sample.groupby(["partition", "modality"]).agg(
            sampled_rows=("identical_features", "size"),
            unchanged_fraction=("identical_features", "mean"),
            mean_real_pooled_norm=("real_pooled_norm", "mean"),
            mean_donor_pooled_norm=("donor_pooled_norm", "mean")))

The shuffled control reassigns whole sequence groups within each outer partition, using one donor for joint auxiliaries. Donor lengths move with donor token arrays; target labels and the receiver's anchor prediction stay fixed. The constant permutation stratum is an explicit project choice. If a partition cannot be deranged, execution stops rather than silently keeping unchanged inputs.

The random-feature control uses the same array dimensions and correction architecture. Feature-wise means/scales come only from training tokens; repeated copies of one group receive the same deterministic random features. Length and shape information remain available to this control. Neither null control establishes that all biological information has been removed, and a negative correction result does not prove that the auxiliary contains no useful information. [Mina's plan, A.2](../../docs/Biological_Fusion_Plan.docx).

The correction run uses `CONTROL_SEED + SEED`: each training seed also changes the shuffled/random realization. Variation across the null-control scores therefore includes control variation as well as model initialization. This is a project setting, not a requirement from Mina. The present repeats do not separately estimate those two sources of variation.

In [18]:
def fit_corrections(opened, benchmark, bundle, selected, final_anchor, oof, fold_artifacts):
    streams, records, groups, positions, split_identity = opened[bundle]
    row_ids = tuple(streams.rows.index)
    offsets = correction.correction_offsets(oof, final_anchor, row_ids=row_ids, split_positions=positions)
    sources = {"final_anchor_sha256": final_anchor.sha256,
               "oof_artifact_sha256": [artifact.sha256 for artifact in fold_artifacts]}
    for auxiliary, modalities in auxiliary_sets(selected["modality"]).items():
        controls = control_specs(streams, positions, groups, modalities, CONTROL_SEED + SEED)
        for control_name, control in controls.items():
            destination = (benchmark_root(benchmark) / "corrections" / bundle / f"seed{SEED}"
                           / auxiliary / control_name)
            contract = {"study": STUDY, "bundle": bundle, "auxiliary": auxiliary, "training_seed": SEED,
                        "outer_split": split_identity, "selection_sha256": selected["selection_sha256"],
                        "offset_sources": sources, "control": {"kind": control_name,
                        "specification_sha256": correction._digest(control)}}
            features = correction.pooled_features(
                streams, np.arange(len(streams.rows)), modalities,
                groups=groups, control=control, split_positions=positions,
            )
            correction.run_ridge(
                features, streams.labels, offsets, row_ids=row_ids, split_positions=positions,
                alphas=RIDGE_ALPHAS, output_dir=destination / "ridge",
                input_contract={**contract, "representation": "native-token-mean-concatenation"},
                producer=PRODUCER,
            )
            del features
            correction.run_neural(
                streams, split_positions=positions, row_ids=row_ids, groups=groups,
                modalities=modalities, layout="native" if len(modalities) == 1 else "codon",
                offsets=offsets, config=replace(BASE_CONFIG, seed=SEED), output_dir=destination / "textcnn",
                cache_records=records, input_contract=contract, device=DEVICE, producer=PRODUCER,
                role="correction", control=control,
            )

---

## Execute one benchmark and one training seed

A job handles both encoder bundles. Seed 0 establishes six unique anchor candidates and publishes two validation selections; seeds 1 and 2 require those selections. The supplied-split CoV-Vac and fungal jobs reuse five compatible Newton candidates and infer their validation predictions from saved best weights. NT-grid is absent from that panel and is fitted anew. Grouped stability instead reuses the six compatible singles from the separately completed grouped-stability panel; its earlier supplied-split results are ineligible. Missing or incompatible Newton evidence holds reuse instead of silently replacing a completed fit.

Each seed has **18 ridge corrections and 18 TextCNN corrections**: two bundles × three auxiliary choices × three controls, for each family. Each distinct selected anchor also needs five new OOF fits per seed. A selected NT-grid anchor needs its final fit for later seeds; grouped stability reuses that panel's corresponding final anchors for every seed. If bundles select the same predictor, they share its final and OOF artifacts.

Across three seeds, a supplied-split benchmark therefore needs **70–87 new neural fits plus 54 ridge fits** with compatible Newton reuse. After its prerequisite panel is complete, grouped stability needs **69–84 new neural fits plus 54 ridge fits**. These bounds allow one or two distinct winners; every ridge fit tries the declared five-alpha grid. Counts describe fits, not runtime. Validation inference, cache checks and feature preparation add work. Completed compatible artifacts are reused and incompatible artifacts are preserved.

`USE_NEWTON_ANCHORS=False` is an explicit separate no-reference mode: it would need 77–94 new neural fits per benchmark, rather than silently replacing missing evidence.

No embeddings are extracted and no supplied targets change. New artifacts stay in this study's result root; Newton source fits and weights remain in their original tree. Newton resource limits and submissions belong to the execution recipe.


In [19]:
def run_study():
    require_training_permission()
    if SEED != SELECTION_SEED:
        for bundle in BUNDLES:
            read_selection(BENCHMARK, bundle)  # Refuse missing seed-0 evidence before opening caches.
    verification = token_embeddings.CacheVerification()
    with ExitStack() as resources:
        opened = {
            bundle: resources.enter_context(study_inputs.open_streams(
                BENCHMARK, mapping, DATA_ROOT, CACHE_ROOT,
                stability_split=GROUPED_SPLIT_PATH, verification=verification,
            ))
            for bundle, mapping in BUNDLES.items()
        }
        selections = prepare_selections(opened, BENCHMARK)
        selection_hashes = {bundle: choice["selection_sha256"] for bundle, choice in selections.items()}
        anchor_results = {}
        for candidate in dict.fromkeys(choice["key"] for choice in selections.values()):
            spec = CANDIDATES[candidate]
            streams, records, groups, positions, split_identity = opened[spec["bundle"]]
            final_anchor = fit_anchor(opened, BENCHMARK, candidate, SEED)
            plans = correction.fold_plan(
                positions["train"], groups, n_splits=N_FOLDS, seed=SPLIT_SEED,
                inner_seed=INNER_SEED, stop_fraction=INNER_STOP_FRACTION,
            )
            oof, fold_artifacts = correction.run_anchor_oof(
                streams, plans=plans, row_ids=tuple(streams.rows.index), groups=groups,
                train_positions=positions["train"], validation_positions=positions["val"],
                test_positions=positions["test"],
                output_dir=benchmark_root(BENCHMARK) / "oof" / candidate / f"seed{SEED}",
                modalities=(spec["modality"],), layout=spec["layout"], config=replace(BASE_CONFIG, seed=SEED),
                cache_records=records, input_contract={"study": STUDY, "candidate": candidate,
                    "outer_split": split_identity, "conditional_selection": selection_hashes,
                    "fold_seed": SPLIT_SEED, "inner_seed": INNER_SEED,
                    "inner_stop_fraction": INNER_STOP_FRACTION},
                device=DEVICE, producer=PRODUCER,
            )
            anchor_results[candidate] = (final_anchor, oof, fold_artifacts)
        for bundle, selected in selections.items():
            fit_corrections(opened, BENCHMARK, bundle, selected, *anchor_results[selected["key"]])

In [20]:
if READ_SAVED_RESULTS:
    print("Training skipped: reading saved evidence.")
elif RUN_TRAINING:
    run_study()
    display(saved_coverage(benchmark_root(BENCHMARK)))

Training skipped: reading saved evidence.


## Compare matched predictions across the three training seeds

The optional report below requires every declared correction and its matching final anchor for seeds 0, 1 and 2. It calculates per-seed test Spearman, the arithmetic mean and the sample standard deviation across seeds. It averages scores, not predictions; a prediction ensemble would answer a different question. All comparisons retain the fixed seed-0 selection.

The report binds the consumed prediction and selection files by hash and writes a new identified analysis. Reading mode alone does not calculate it. Paired bootstrap resamples whole test-sequence groups, using the same draw for all models and seeds and averaging seed scores inside each draw. Its interval measures test-sample uncertainty conditional on the fitted seeds; it is separate from seed standard deviation. Every real-input correction is compared with its fixed anchor and its matched shuffled/random controls. Ridge-versus-TextCNN is an optional, prospectively declared contrast and is off by default. Intervals are unadjusted for multiple comparisons; any undefined resampled scores are counted and the interval is withheld rather than silently dropping them.

In [21]:
def read_prediction_record(path):
    path = Path(path)
    record = json.loads(path.read_text(encoding="utf-8"))
    if record.get("contract") != STUDY:
        raise ValueError(f"Different prediction contract: {path}")
    if record.get("kind") == "reference_anchor":
        correction.reference_artifacts(
            record, GROUPED_REFERENCE_ROOT if record.get("reference_location") == "grouped_stability" else REFERENCE_ROOT)
    for descriptor in (() if record.get("kind") == "reference_anchor" else record.get("artifacts", {}).values()):
        source = path.parent / descriptor["file"]
        if source.parent != path.parent or study_selection.file_sha256(source) != descriptor["sha256"]:
            raise ValueError(f"Changed fitted source: {source}")
    if record["kind"] == "neural" and set(record.get("artifacts", {})) != {"fit", "weights"}:
        raise ValueError("Neural predictions lack their completed fit/weights binding.")
    if record["kind"] == "neural":
        fit_path = path.parent / record["artifacts"]["fit"]["file"]
        fit = json.loads(fit_path.read_text(encoding="utf-8"))
        test = record["predictions"]["test"]
        if (fit.get("identity") != record["identity"] or fit.get("test_truth") != test["truth"]
                or fit.get("test_prediction") != test["values"]):
            raise ValueError("Prediction summary differs from its fitted source.")
    elif record["kind"] == "reference_anchor":
        if record.get("inference_identity", {}).get("inputs", {}).get("role") != "anchor":
            raise ValueError("Reference prediction summary lacks its A2 inference contract.")
    elif record["kind"] != "ridge" or not record.get("fitted") or not record.get("candidates"):
        raise ValueError("Unknown or incomplete saved correction family.")
    for split in ("val", "test"):
        entry = record["predictions"][split]
        values, truth = np.asarray(entry["values"]), np.asarray(entry["truth"])
        if (values.shape != truth.shape or values.shape != (len(entry["row_ids"]),)
                or not np.isfinite(values).all() or not np.isfinite(truth).all()):
            raise ValueError("Saved prediction rows are incomplete or nonfinite.")
    return record

In [22]:
def report_contrasts(anchor_modality):
    contrasts = []
    for auxiliary in auxiliary_sets(anchor_modality):
        for family in CORRECTION_FAMILIES:
            real = f"{auxiliary}/real/{family}"
            for label, other in (("correction vs anchor", "anchor"),
                                 ("real vs shuffled", f"{auxiliary}/shuffle/{family}"),
                                 ("real vs random", f"{auxiliary}/random/{family}")):
                contrasts.append({"comparison": f"{auxiliary}/{family}: {label}",
                                  "first": real, "second": other})
        if COMPARE_CORRECTION_FAMILIES:
            contrasts.append({"comparison": f"{auxiliary}: TextCNN vs ridge",
                              "first": f"{auxiliary}/real/textcnn", "second": f"{auxiliary}/real/ridge"})
    return contrasts


def collect_report_panel(benchmark, bundle):
    selected = read_selection(benchmark, bundle)
    selection_path = benchmark_root(benchmark) / f"selection_{bundle}.json"
    sources = {str(selection_path.relative_to(RESULTS_ROOT)): study_selection.file_sha256(selection_path)}
    models, rows, reference = {}, [], None
    for seed in TRAINING_SEEDS:
        anchor_path = anchor_folder(benchmark, selected["key"], seed) / "predictions.json"
        anchor = read_prediction_record(anchor_path)
        anchor_identity = anchor.get("inference_identity", anchor["identity"])
        if (anchor_identity["config"]["seed"] != seed
                or anchor_identity["inputs"]["protocol"]["candidate"] != selected["key"]):
            raise ValueError("Anchor evidence differs from the requested seed/configuration.")
        anchor_sha = study_selection.file_sha256(anchor_path)
        test = anchor["predictions"]["test"]
        group_ids = [anchor_identity["inputs"]["groups"][index] for index in test["positions"]]
        current = {"row_ids": test["row_ids"], "truth": test["truth"], "groups": group_ids}
        if reference is not None and reference != current:
            raise ValueError("Test rows, targets or sequence groups differ across training seeds.")
        reference = current
        paths = {"anchor": anchor_path}
        for auxiliary in auxiliary_sets(selected["modality"]):
            for control in CONTROL_KINDS:
                for family in CORRECTION_FAMILIES:
                    paths[f"{auxiliary}/{control}/{family}"] = (
                        benchmark_root(benchmark) / "corrections" / bundle / f"seed{seed}"
                        / auxiliary / control / family / "predictions.json"
                    )
        for name, path in paths.items():
            saved = anchor if name == "anchor" else read_prediction_record(path)
            entry = saved["predictions"]["test"]
            if entry["row_ids"] != reference["row_ids"] or entry["truth"] != reference["truth"]:
                raise ValueError("Correction and anchor test observations differ.")
            if name != "anchor":
                identity = saved["identity"]
                protocol = identity["inputs"]["protocol"] if saved["kind"] == "neural" else identity["protocol"]
                if protocol.get("training_seed") != seed or protocol.get("bundle") != bundle:
                    raise ValueError("Correction belongs to another training seed or encoder bundle.")
                if (protocol["offset_sources"]["final_anchor_sha256"] != anchor_sha
                        or protocol["selection_sha256"] != selected["selection_sha256"]):
                    raise ValueError("Correction used a different anchor or validation selection.")
            score = float(train.score("regression", np.asarray(entry["truth"], dtype=np.float32),
                                      np.asarray(entry["values"])))
            if not np.isfinite(score):
                raise ValueError(f"Undefined test Spearman: {path}; no aggregate is published.")
            rows.append({"benchmark": benchmark, "bundle": bundle, "model": name,
                         "seed": seed, "spearman": score})
            models.setdefault(name, {})[seed] = {"data": {
                "test_truth": entry["truth"], "test_prediction": entry["values"],
            }}
            sources[str(path.relative_to(RESULTS_ROOT))] = study_selection.file_sha256(path)
    panel = {"models": models, "seeds": tuple(TRAINING_SEEDS),
             "comparisons": report_contrasts(selected["modality"])}
    return rows, sources, reference, panel

In [23]:
def calculate_three_seed_report(benchmark):
    collected = {bundle: collect_report_panel(benchmark, bundle) for bundle in BUNDLES}
    sources = {path: sha for _, source_map, _, _ in collected.values() for path, sha in source_map.items()}
    identity = {
        "study": STUDY, "benchmark": benchmark, "seeds": list(TRAINING_SEEDS), "sources": sources,
        "statistic": "mean of per-seed Spearman; sample seed SD; paired test-sequence-group bootstrap",
        "groups": {bundle: correction._digest(reference["groups"])
                   for bundle, (_, _, reference, _) in collected.items()},
        "contrasts": {bundle: panel["comparisons"] for bundle, (_, _, _, panel) in collected.items()},
        "draws": BOOTSTRAP_DRAWS, "bootstrap_seed": BOOTSTRAP_SEED,
        "scoring_target_precision": "float32",
        "source_sha256": {Path(module.__file__).name: study_selection.file_sha256(Path(module.__file__))
                          for module in (train, swap_reporting)},
    }
    notebook_source = json.loads((CODE_DIR / "stage4" / "anchor_correction.ipynb").read_text(encoding="utf-8"))
    identity["source_sha256"]["anchor_correction.ipynb.code"] = correction._digest([
        "".join(cell["source"]) for cell in notebook_source["cells"] if cell["cell_type"] == "code"
    ])
    destination = benchmark_root(benchmark) / "analyses" / f"seed-summary-{correction._digest(identity)[:16]}.json"
    if destination.exists():
        saved = json.loads(destination.read_text(encoding="utf-8"))
        if (saved.get("identity") != identity
                or saved.get("analysis_sha256") != correction._digest(saved.get("analysis"))):
            raise ValueError("An identified report is incomplete or incompatible; preserve it.")
        analysis = saved["analysis"]
        return pd.DataFrame(analysis["scores"]), pd.DataFrame(analysis["summary"]), destination
    scores = pd.DataFrame([row for rows, _, _, _ in collected.values() for row in rows])
    summary = scores.groupby(["benchmark", "bundle", "model"], sort=False)["spearman"].agg(
        mean="mean", seed_sd="std", seeds="count",
    ).reset_index()
    if not (summary["seeds"] == len(TRAINING_SEEDS)).all():
        raise ValueError("The requested seed cohort is incomplete.")
    paired = {
        bundle: swap_reporting.calculate(
            panel, truth=np.asarray(reference["truth"], dtype=np.float32),
            groups=np.asarray(reference["groups"]), task="regression", context={},
            draws=BOOTSTRAP_DRAWS, bootstrap_seed=BOOTSTRAP_SEED,
        )
        for bundle, (_, _, reference, panel) in collected.items()
    }
    # Refuse a producer file changed while the arrays were being collected/scored.
    for path, sha in sources.items():
        if study_selection.file_sha256(Path(RESULTS_ROOT) / path) != sha:
            raise ValueError("A consumed prediction/selection artifact changed during reporting.")
    analysis = {"scores": scores.to_dict("records"), "summary": summary.to_dict("records"), "paired": paired}
    correction._publish(destination, {"identity": identity, "analysis": analysis,
                                      "analysis_sha256": correction._digest(analysis),
                                      "producer": run_identity.producer_record(
                                          device="cpu", repository=PROJECT_DIR, packages=("numpy", "scipy"))})
    return scores, summary, destination

In [24]:
def read_saved_reports(benchmark):
    reports = []
    for path in sorted((benchmark_root(benchmark) / "analyses").glob("seed-summary-*.json")):
        saved = json.loads(path.read_text(encoding="utf-8"))
        if (saved.get("identity", {}).get("study") != STUDY
                or saved["identity"].get("benchmark") != benchmark
                or saved["identity"].get("seeds") != list(TRAINING_SEEDS)
                or not saved["identity"].get("sources")
                or saved.get("analysis_sha256") != correction._digest(saved.get("analysis"))):
            raise ValueError(f"Incomplete saved statistical report: {path}")
        for relative, sha in saved["identity"]["sources"].items():
            source = (Path(RESULTS_ROOT) / relative).resolve()
            if (not source.is_relative_to(Path(RESULTS_ROOT).resolve())
                    or study_selection.file_sha256(source) != sha):
                raise ValueError(f"Changed source of saved report: {path}")
        reports.append((path, saved))
    return reports

In [25]:
if RUN_ANALYSIS:
    if not READ_SAVED_RESULTS or RUN_TRAINING or RUN_DIAGNOSTICS:
        raise ValueError("Saved-prediction analysis must be a separate read-results execution.")
    per_seed, three_seed_summary, report_path = calculate_three_seed_report(BENCHMARK)
    display(per_seed)
    display(three_seed_summary)
    print(f"Saved comparison summary: {report_path}")
else:
    print("Three-seed scores and paired intervals are not recalculated.")

for report_path, saved_report in read_saved_reports(BENCHMARK):
    print(f"Reading saved report: {report_path.name}")
    display(pd.DataFrame(saved_report["analysis"]["summary"]))
    paired_rows = [{"bundle": bundle, **row}
                   for bundle, comparison in saved_report["analysis"]["paired"].items()
                   for row in comparison["seed_summary"]]
    display(pd.DataFrame(paired_rows))


Three-seed scores and paired intervals are not recalculated.


## Can the retained evidence explain a correction result?

The OOF anchors use about 64% of the outer-training groups per fit: roughly four fifths remain after holding out one fold, then four fifths of those enter fitting rather than inner stopping. Actual proportions are reported above. The final anchor uses all outer-training rows. A correction can therefore learn errors characteristic of a smaller training set rather than errors that persist for the final anchor. Inspect fold residual spread and best epochs alongside final-anchor validation residuals. Similar summaries support transfer of the correction target; discrepancies challenge it. Because OOF training groups and outer-validation groups differ, this is descriptive evidence, not a matched causal comparison. Predicting the same outer-validation rows with every fold model would be a separate, explicitly authorized inference workload; it is not added here.

Ridge and TextCNN remain two complementary probes, not a clean linear-versus-nonlinear ablation. Ridge tunes validation MSE, while TextCNN selects corrected validation Spearman; pooling, positional access, alignment and capacity also differ. A ridge optimum at a grid boundary or an uninformative neural learning curve can motivate later tuning review, but never expands the grid or budget automatically. Correction magnitudes below are calculated on validation rows with the same final anchor. They can reveal a largely constant recalibration or large unstable updates without establishing which biological feature is responsible.

`RUN_DIAGNOSTICS` reads retained prediction/fit JSONs, summarizes OOF and validation errors and histories, and saves a separate report. It performs no fitting, model loading, test-score calculation or token-cache access. Fit JSON hashes bind the observed histories; this metadata inspection does not verify best-weight bytes. Missing/incompatible evidence is listed explicitly. Reading saved reports only checks their recorded report integrity, not the current presence of every upstream input.

In [26]:
def inspect_saved_correction_evidence():
    residuals, curves, ridge_grid, updates, random_scales, issues = [], [], [], [], [], []
    sources, records, anchors = {}, {}, {}
    roots = selection_evidence_roots()

    def record_source(path):
        path = Path(path).resolve()
        choices = [key for key, root in roots.items() if path.is_relative_to(root.resolve())]
        if not choices:
            raise ValueError("Diagnostic evidence escaped its declared roots.")
        key = max(choices, key=lambda value: len(roots[value].resolve().parts))
        name = key + "/" + path.relative_to(roots[key].resolve()).as_posix()
        sources[name] = study_selection.file_sha256(path)
        return sources[name]

    for path in sorted(benchmark_root(BENCHMARK).glob("**/predictions.json")):
        model = path.parent.relative_to(benchmark_root(BENCHMARK)).as_posix()
        try:
            sha = record_source(path)
            record = json.loads(path.read_text(encoding="utf-8"))
            if record.get("contract") != STUDY or record.get("kind") not in ("neural", "ridge", "reference_anchor"):
                raise ValueError("Unsupported prediction contract/family.")
            identity = record.get("inference_identity", record["identity"])
            split = "test" if model.startswith("oof/") else "val"
            entry = record["predictions"][split]
            summary = correction.evidence.residual_summary(entry)
            population = "OOF held-out training groups" if split == "test" else "outer validation"
            fit = None
            if record["kind"] in ("neural", "reference_anchor"):
                descriptor = record["artifacts"]["fit"]
                if record["kind"] == "reference_anchor":
                    reference = GROUPED_REFERENCE_ROOT if record.get("reference_location") == "grouped_stability" else REFERENCE_ROOT
                    fit_path = (Path(reference) / descriptor["reference_relative"]).resolve()
                    if not fit_path.is_relative_to(Path(reference).resolve()):
                        raise ValueError("Reference fit path escaped its root.")
                else:
                    if Path(descriptor["file"]).name != descriptor["file"]:
                        raise ValueError("Fit descriptor must name a sibling file.")
                    fit_path = path.parent / descriptor["file"]
                if record_source(fit_path) != descriptor["sha256"]:
                    raise ValueError("Fit JSON differs from the prediction record.")
                fit = json.loads(fit_path.read_text(encoding="utf-8"))
                if fit.get("identity") != record["identity"]:
                    raise ValueError("Fit and prediction identities differ.")
                for epoch in fit.get("history", []):
                    curves.append({"model": model, "epoch": epoch["epoch"],
                                   "train_loss": epoch["train_loss"], "val_score": epoch["val_score"],
                                   "best_epoch": fit["best_epoch"]})
                summary.update(best_epoch=fit.get("best_epoch"), completed_epochs=fit.get("epochs"),
                               saved_validation_score=fit.get("val_score"))
            else:
                candidates = record["candidates"]
                alphas = [candidate["alpha"] for candidate in candidates]
                if record["selected_alpha"] not in alphas:
                    raise ValueError("Selected ridge alpha is absent from its grid.")
                summary.update(selected_alpha=record["selected_alpha"],
                               alpha_at_grid_edge=record["selected_alpha"] in (alphas[0], alphas[-1]))
                ridge_grid.extend({"model": model, **candidate} for candidate in candidates)
            residuals.append({"model": model, "population": population, **summary})
            records[model] = (record, identity)
            if model.startswith("anchors/"):
                anchors[sha] = record
            control = identity.get("inputs", {}).get("control", {})
            if control.get("kind") == "random":
                for modality, statistics in control["statistics"].items():
                    means, scales = np.asarray(statistics["mean"]), np.asarray(statistics["scale"])
                    random_scales.append({"model": model, "modality": modality,
                        "dimensions": len(scales), "constant_dimensions": int(np.sum(scales == 0)),
                        "training_mean_norm": float(np.linalg.norm(means)),
                        "training_scale_norm": float(np.linalg.norm(scales))})
        except (KeyError, TypeError, ValueError, OSError) as error:
            issues.append({"model": model, "issue": str(error)})

    for model, (record, identity) in records.items():
        if not model.startswith("corrections/"):
            continue
        try:
            protocol = identity.get("inputs", {}).get("protocol", identity.get("protocol", {}))
            anchor = anchors[protocol["offset_sources"]["final_anchor_sha256"]]
            entry, baseline = record["predictions"]["val"], anchor["predictions"]["val"]
            if entry["row_ids"] != baseline["row_ids"] or entry["truth"] != baseline["truth"]:
                raise ValueError("Correction and bound anchor validation rows/targets differ.")
            values = np.asarray(entry["values"]) - np.asarray(baseline["values"])
            updates.append({"model": model, "rows": len(values), "correction_mean": float(values.mean()),
                            "correction_sd": float(values.std()),
                            "correction_rms": float(np.sqrt(np.mean(values ** 2)))})
        except (KeyError, TypeError, ValueError) as error:
            issues.append({"model": model, "issue": f"Matched validation update unavailable: {error}"})
    if not sources:
        raise ValueError("No saved prediction evidence exists; this inspection never creates fits.")
    # Bind every consumed JSON again before publication; do not aggregate a moving producer.
    for name, expected in sources.items():
        root, relative = name.split("/", 1)
        if study_selection.file_sha256(roots[root] / relative) != expected:
            raise ValueError("Saved diagnostic evidence changed during inspection.")
    analysis = {"residuals": residuals, "learning_curves": curves, "ridge_grid": ridge_grid,
                "validation_updates": updates, "random_scales": random_scales, "issues": issues}
    return save_decision_report("saved-evidence", analysis, {"sources": sources})

In [27]:
if RUN_DIAGNOSTICS:
    print(f"Saved model-evidence inspection: {inspect_saved_correction_evidence()}")
for path, report in read_decision_reports("saved-evidence"):
    print(f"Recorded prediction snapshot: {path.name}")
    for name in ("residuals", "validation_updates", "ridge_grid", "random_scales", "issues"):
        table = pd.DataFrame(report["analysis"][name])
        if not table.empty:
            print(name.replace("_", " "))
            display(table)
    print("Learning curves are retained in the report; choose a model below to inspect them.")

In [28]:
# Explicit model selection keeps the plot readable across a large correction panel.
DIAGNOSTIC_MODEL = None  # Example: "oof/ntv2_native/seed0/fold0".
if DIAGNOSTIC_MODEL is not None:
    from matplotlib import pyplot as plt

    for path, report in read_decision_reports("saved-evidence"):
        history = pd.DataFrame(report["analysis"]["learning_curves"])
        if history.empty:
            continue
        history = history.loc[history["model"] == DIAGNOSTIC_MODEL]
        if history.empty:
            continue
        figure, axes = plt.subplots(1, 2, figsize=(9, 3))
        axes[0].plot(history["epoch"], history["train_loss"])
        axes[0].set(xlabel="Epoch (zero-based)", ylabel="Training MSE")
        axes[1].plot(history["epoch"], history["val_score"])
        axes[1].set(xlabel="Epoch (zero-based)", ylabel="Stopping-set Spearman")
        for axis in axes:
            axis.axvline(history["best_epoch"].iloc[0], color="grey", linestyle="--")
        figure.suptitle(DIAGNOSTIC_MODEL)
        figure.tight_layout()
        plt.show()
        plt.close(figure)

## What would the comparisons establish?

An auxiliary earns further investigation when its real-input correction improves the matched fixed anchor consistently and also outperforms the shuffled and random-feature controls. Improvement only over the anchor may reflect added capacity, optimization, retained length information or a weak control. Ridge and TextCNN provide two views of accessible predictive information; their different pooling and positional access prevent assigning a difference solely to nonlinear modeling.

A null correction can reflect an adequate anchor, insufficient correction capacity, noisy OOF targets or a mismatch between training-fold and final-anchor errors. It does not establish biological redundancy. Gate and attention experiments should follow the remaining question, rather than treating this notebook as a prerequisite that must produce a positive result. [Connected Stage 4 investigation](README.md); [Mina's protocol](../../docs/Biological_Fusion_Plan.docx).

No findings are prewritten here. Completed runs, per-seed variation, null controls, failures and any subsequently authorized paired intervals should be interpreted together. The source tables and uncertainty limits remain separate from these implementation checks.